
# Test Case Generation, RAG, And Evaluation Pipeline

This notebook contains the second half of the project. It uses the validated schema from the first notebook, audits the existing mutation dataset, builds a multi-stage retrieval pipeline, evaluates retrieval quality, synthesizes richer test categories, analyzes PL/SQL code structure, and produces rubric-style scoring artifacts.



## Testing Notes

This notebook includes explicit RAG evaluation checks and downstream pipeline checks. The goal is not only to generate test cases but also to verify that the retrieval and generation logic are behaving correctly on the current assignment.


In [ ]:

from pathlib import Path
import json
import math
import re
from collections import Counter

ROOT = Path(r"C:\amrita_uni\s6\NLP\project\Rubric-based-evaluation-of-PL-SQL-code\Rubric-based-evaluation-of-PL-SQL-code")
SCHEMA_ARTIFACT_DIR = ROOT / "artifacts" / "schema_output"
ARTIFACT_DIR = ROOT / "artifacts" / "testcase_output"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
schema = json.loads((SCHEMA_ARTIFACT_DIR / "schema.json").read_text(encoding="utf-8"))
ddl_text = (SCHEMA_ARTIFACT_DIR / "ddl.sql").read_text(encoding="utf-8")
prompt_decomposition = json.loads((SCHEMA_ARTIFACT_DIR / "prompt_decomposition.json").read_text(encoding="utf-8"))
schema_confidence = json.loads((SCHEMA_ARTIFACT_DIR / "schema_confidence.json").read_text(encoding="utf-8"))
problem_statement = 'Consider a Bank database which includes the following tables.\n\nACCOUNTS(ac_no, br_no, cust_no, ac_type, bal)\nBRANCHES(br_no, br_name, loc)\nCUSTOMER(cno, cname, c_type)\n\n1. Write a function that accepts a threshold value and a customer number. The program updates the c_type based on the threshold value. If balance > threshold then class A, else class B.\n2. Write a function called CloseBranch that takes two arguments (the branch to be closed and the branch to take over the accounts) and transfers all accounts at the closing branch to the new branch and removes the closing branch.\n3. Write a function that implements a safe withdrawal operation, that only permits a withdraw if there are sufficient funds in the account to cover it.\n'
submission_text = "CREATE OR REPLACE PROCEDURE classify_customer (\n    p_threshold IN NUMBER,\n    p_customer_no IN NUMBER\n) IS\n    v_balance ACCOUNTS.bal%TYPE;\nBEGIN\n    SELECT bal\n    INTO v_balance\n    FROM accounts\n    WHERE cust_no = p_customer_no\n      AND ROWNUM = 1;\n\n    IF v_balance > p_threshold THEN\n        UPDATE customer\n        SET c_type = 'A'\n        WHERE cno = p_customer_no;\n    ELSE\n        UPDATE customer\n        SET c_type = 'B'\n        WHERE cno = p_customer_no;\n    END IF;\nEXCEPTION\n    WHEN NO_DATA_FOUND THEN\n        DBMS_OUTPUT.PUT_LINE('Customer not found');\nEND;\n/\n\nCREATE OR REPLACE PROCEDURE safe_withdraw (\n    p_account_no IN NUMBER,\n    p_amount IN NUMBER\n) IS\n    v_balance ACCOUNTS.bal%TYPE;\nBEGIN\n    SELECT bal\n    INTO v_balance\n    FROM accounts\n    WHERE ac_no = p_account_no\n    FOR UPDATE;\n\n    IF v_balance >= p_amount THEN\n        UPDATE accounts\n        SET bal = bal - p_amount\n        WHERE ac_no = p_account_no;\n        COMMIT;\n    ELSE\n        RAISE_APPLICATION_ERROR(-20001, 'Insufficient funds');\n    END IF;\nEXCEPTION\n    WHEN OTHERS THEN\n        ROLLBACK;\n        RAISE;\nEND;\n/\n"



## Stage 1: Mutation Corpus Audit

This stage uses the existing mutation dataset already present in the folder. It profiles the corpus before retrieval so we can inspect category balance, tag availability, and coverage quality instead of treating the dataset like a black box.


In [2]:

def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-zA-Z_][a-zA-Z0-9_]+", text.lower())

def count_terms(tokens: list[str]) -> dict[str, float]:
    counts = {}
    for token in tokens:
        counts[token] = counts.get(token, 0.0) + 1.0
    return counts

def cosine_similarity(left: dict[str, float], right: dict[str, float]) -> float:
    if not left or not right:
        return 0.0
    numerator = sum(left.get(token, 0.0) * right.get(token, 0.0) for token in left)
    left_norm = math.sqrt(sum(value * value for value in left.values()))
    right_norm = math.sqrt(sum(value * value for value in right.values()))
    if not left_norm or not right_norm:
        return 0.0
    return numerator / (left_norm * right_norm)

def canonical_focus_keyword(token: str) -> str:
    if token.startswith("withdraw"):
        return "withdrawal"
    if token.startswith("branch"):
        return "branch"
    if token.startswith("transfer"):
        return "transfer"
    if token.startswith("threshold"):
        return "threshold"
    if token.startswith("balanc"):
        return "balance"
    if token.startswith("account"):
        return "account"
    if token.startswith("customer"):
        return "customer"
    if token.startswith("except"):
        return "exception"
    return token

def load_mutation_documents() -> list[dict]:
    path = ROOT / "plsql_mutation_rag_dataset.jsonl"
    documents = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            documents.append(json.loads(line))
    return documents

mutation_documents = load_mutation_documents()
corpus_profile = {
    "document_count": len(mutation_documents),
    "category_distribution": Counter(doc.get("category", "unknown") for doc in mutation_documents),
    "avg_tag_count": round(sum(len(doc.get("tags", [])) for doc in mutation_documents) / max(1, len(mutation_documents)), 2),
}
print("Corpus profile:")
print("document_count:", corpus_profile["document_count"])
print("top categories:", corpus_profile["category_distribution"].most_common(10))
print("avg_tag_count:", corpus_profile["avg_tag_count"])


Corpus profile:
document_count: 85
top categories: [('sql', 14), ('control_flow', 9), ('paper_operator', 9), ('declaration', 6), ('exception', 6), ('business_rule', 6), ('parameter', 5), ('transaction', 4), ('cursor', 4), ('dynamic_sql', 4)]
avg_tag_count: 4.0



## Stage 2: Retrieval Query Planning

Before ranking documents, the notebook builds a structured retrieval query from the schema, prompt decomposition, and procedural intents. This makes the RAG stage assignment-aware rather than generic.


In [3]:

def build_retrieval_query(schema: dict, decomposition: dict, problem_text: str) -> dict:
    focus_terms = set()
    for token in tokenize(problem_text):
        canonical = canonical_focus_keyword(token)
        if canonical in {"branch", "transfer", "threshold", "exception", "transaction"}:
            focus_terms.add(canonical)
    intent_proxies = {}
    for intent in schema.get("procedural_intents", []):
        name = intent.get("name")
        if name == "safe_withdrawal":
            intent_proxies[name] = ["transaction", "exception", "boundary"]
            focus_terms.add("transaction")
            focus_terms.add("exception")
        elif name == "branch_transfer":
            intent_proxies[name] = ["branch", "transaction", "transfer"]
            focus_terms.add("branch")
            focus_terms.add("transaction")
        elif name == "threshold_classification":
            intent_proxies[name] = ["threshold", "boundary", "business_rule"]
            focus_terms.add("threshold")
    intent_tags = sorted({
        tag
        for intent in schema.get("procedural_intents", [])
        for tag in intent.get("risk_tags", [])
    })
    query_text = "\n".join(
        [
            problem_text,
            json.dumps(decomposition, indent=2),
            json.dumps(schema.get("procedural_intents", []), indent=2),
        ]
    )
    return {
        "focus_terms": sorted(focus_terms),
        "intent_tags": intent_tags,
        "intent_proxies": intent_proxies,
        "query_text": query_text,
    }

retrieval_query = build_retrieval_query(schema, prompt_decomposition, problem_statement)
print(json.dumps(retrieval_query, indent=2))


{
  "focus_terms": [
    "branch",
    "exception",
    "threshold",
    "transaction",
    "transfer"
  ],
  "intent_tags": [
    "balance",
    "boundary",
    "branch",
    "comparison",
    "customer",
    "foreign_key",
    "threshold",
    "transaction",
    "transfer",
    "withdrawal"
  ],
  "intent_proxies": {
    "threshold_classification": [
      "threshold",
      "boundary",
      "business_rule"
    ],
    "safe_withdrawal": [
      "transaction",
      "exception",
      "boundary"
    ],
    "branch_transfer": [
      "branch",
      "transaction",
      "transfer"
    ]
  },
  "query_text": "Consider a Bank database which includes the following tables.\n\nACCOUNTS(ac_no, br_no, cust_no, ac_type, bal)\nBRANCHES(br_no, br_name, loc)\nCUSTOMER(cno, cname, c_type)\n\n1. Write a function that accepts a threshold value and a customer number. The program updates the c_type based on the threshold value. If balance > threshold then class A, else class B.\n2. Write a function c


## Stage 3: Hybrid Retrieval, Reranking, And Diversity Selection

This RAG pipeline now has three steps:

1. lexical relevance against the planned query
2. procedural-risk boosting using the schema intents
3. diversity-aware selection so multiple fault families appear in the final set


In [4]:

def document_feature_text(doc: dict) -> str:
    return " ".join(
        [
            doc.get("mutation_name", ""),
            doc.get("content", ""),
            doc.get("why_it_matters", ""),
            doc.get("expected_observable_effect", ""),
            doc.get("category", ""),
            " ".join(doc.get("tags", [])),
        ]
    )

def score_documents(documents: list[dict], query: dict) -> list[dict]:
    query_tokens = tokenize(query["query_text"])
    query_vector = count_terms(query_tokens)
    focus_weights = {
        "branch": 3.5,
        "transfer": 3.5,
        "threshold": 2.5,
        "exception": 1.8,
        "transaction": 3.0,
    }
    ranked = []
    for doc in documents:
        text = document_feature_text(doc)
        doc_tokens = tokenize(text)
        doc_vector = count_terms(doc_tokens)
        doc_terms = {canonical_focus_keyword(token) for token in doc_tokens}
        lexical_score = cosine_similarity(query_vector, doc_vector)
        focus_score = sum(focus_weights[term] for term in query["focus_terms"] if term in doc_terms)
        tag_score = sum(0.75 for tag in query["intent_tags"] if tag in doc.get("tags", []))
        category_score = 1.2 if doc.get("category") in {"transaction", "business_rule", "exception", "control_flow"} else 0.0
        ranked.append(
            {
                "id": doc.get("id"),
                "mutation_name": doc.get("mutation_name"),
                "category": doc.get("category"),
                "tags": doc.get("tags", []),
                "why_it_matters": doc.get("why_it_matters", ""),
                "expected_observable_effect": doc.get("expected_observable_effect", ""),
                "lexical_score": lexical_score,
                "focus_score": focus_score,
                "tag_score": tag_score,
                "category_score": category_score,
                "total_score": lexical_score + focus_score + tag_score + category_score,
            }
        )
    return sorted(ranked, key=lambda item: item["total_score"], reverse=True)

def select_diverse_mutations(ranked: list[dict], focus_terms: list[str], top_k: int = 8) -> list[dict]:
    selected = []
    used_ids = set()
    target_terms = ["transaction", "branch", "transfer", "threshold", "exception"]
    for target in target_terms:
        if target not in focus_terms and target != "exception":
            continue
        for item in ranked:
            doc_terms = {canonical_focus_keyword(token) for token in tokenize(item["mutation_name"] + " " + " ".join(item["tags"]))}
            if target in doc_terms and item["id"] not in used_ids:
                selected.append(item)
                used_ids.add(item["id"])
                break
    category_quota = Counter()
    for item in ranked:
        if item["id"] in used_ids:
            continue
        if category_quota[item["category"]] >= 2:
            continue
        selected.append(item)
        used_ids.add(item["id"])
        category_quota[item["category"]] += 1
        if len(selected) >= top_k:
            break
    return selected[:top_k]

ranked_mutations = score_documents(mutation_documents, retrieval_query)
retrieved_mutations = select_diverse_mutations(ranked_mutations, retrieval_query["focus_terms"], top_k=8)
for item in retrieved_mutations:
    print(item["id"], item["mutation_name"], "|", item["category"], "|", round(item["total_score"], 3))


MUT-043 COMMIT Removal | transaction | 5.03
MUT-051 SQLCODE Branch Mutation | exception | 7.47
MUT-074 Threshold Tightening | business_rule | 5.399
MUT-050 Raise Removed | exception | 6.663
MUT-009 Call-Site Boundary Value | parameter | 6.962
MUT-014 Greater-Than to Greater-Or-Equal | relational | 6.949
MUT-020 IF Branch Removal | control_flow | 5.667
MUT-028 GOTO Target Mutation | control_flow | 5.548



## Stage 4: RAG Quality Evaluation

This is the notebook-level RAG test stage. It measures whether the retrieved set covers the assignment intents well enough to justify using it for test generation.


In [5]:

def evaluate_rag_quality(retrieved: list[dict], query: dict) -> dict:
    retrieved_terms = set()
    retrieved_categories = Counter()
    for item in retrieved:
        retrieved_categories[item["category"]] += 1
        terms = {canonical_focus_keyword(token) for token in tokenize(item["mutation_name"] + " " + " ".join(item["tags"]))}
        retrieved_terms.update(terms)
    intent_hits = {}
    for intent_name, proxies in query.get("intent_proxies", {}).items():
        intent_hits[intent_name] = any(proxy in retrieved_terms or proxy in retrieved_categories for proxy in proxies)
    coverage = sum(1 for hit in intent_hits.values() if hit) / max(1, len(intent_hits))
    return {
        "intent_proxy_coverage": round(coverage, 3),
        "intent_hits": intent_hits,
        "retrieved_categories": dict(retrieved_categories),
        "retrieved_count": len(retrieved),
        "diverse_category_count": len(retrieved_categories),
    }

rag_quality = evaluate_rag_quality(retrieved_mutations, retrieval_query)
print(json.dumps(rag_quality, indent=2))


{
  "intent_proxy_coverage": 1.0,
  "intent_hits": {
    "threshold_classification": true,
    "safe_withdrawal": true,
    "branch_transfer": true
  },
  "retrieved_categories": {
    "transaction": 1,
    "exception": 2,
    "business_rule": 1,
    "parameter": 1,
    "relational": 1,
    "control_flow": 2
  },
  "retrieved_count": 8,
  "diverse_category_count": 6
}



## Stage 5: Test Intent Planning And Synthesis

Instead of creating only basic inserts, the notebook now synthesizes several categories:

- normal cases
- boundary cases
- negative cases
- mutation-driven cases
- exception-oriented scenarios


In [6]:

def sql_literal(value):
    if value is None:
        return "NULL"
    if isinstance(value, bool):
        return "1" if value else "0"
    if isinstance(value, (int, float)):
        return str(value)
    return "'" + str(value).replace("'", "''") + "'"

def build_value(attribute_name: str, type_hint: str, row_index: int) -> object:
    upper = type_hint.upper()
    lower = attribute_name.lower()
    if "NUMBER" in upper or "INT" in upper or "FLOAT" in upper:
        if any(token in lower for token in ["bal", "sal", "price", "mark"]):
            return float(1000 + row_index * 150)
        return row_index
    if "DATE" in upper:
        return f"2024-01-0{row_index}"
    if "type" in lower:
        return "A"
    if "name" in lower:
        return f"{attribute_name}_{row_index}"
    if "loc" in lower:
        return f"Location_{row_index}"
    if "email" in lower:
        return f"user{row_index}@example.com"
    return f"{attribute_name}_{row_index}"

def build_seed_rows(schema: dict) -> dict:
    seeds = {}
    for entity_index, entity in enumerate(schema.get("entities", []), start=1):
        rows = []
        for row_index in range(1, 4):
            row = {}
            for attribute in entity.get("attributes", []):
                row[attribute["name"]] = build_value(attribute["name"], attribute.get("type_hint", "VARCHAR2(255)"), row_index)
            for pk_pos, key in enumerate(entity.get("primary_key", []), start=1):
                row[key] = entity_index * 100 + row_index * 10 + pk_pos
            rows.append(row)
        seeds[entity["name"]] = rows
    for rel in schema.get("relationships", []):
        parents = seeds.get(rel.get("to_entity"), [])
        children = seeds.get(rel.get("from_entity"), [])
        for idx, child in enumerate(children):
            if not parents:
                continue
            parent = parents[min(idx, len(parents) - 1)]
            for from_col, to_col in zip(rel.get("from_columns", []), rel.get("to_columns", [])):
                if to_col in parent:
                    child[from_col] = parent[to_col]
    return seeds

def render_insert(table_name: str, row: dict) -> str:
    columns = ", ".join(row.keys())
    values = ", ".join(sql_literal(value) for value in row.values())
    return f"INSERT INTO {table_name} ({columns}) VALUES ({values});"

def plan_test_intents(schema: dict, retrieved: list[dict]) -> list[dict]:
    intents = []
    for procedural_intent in schema.get("procedural_intents", []):
        intents.append(
            {
                "name": procedural_intent["name"],
                "risk_tags": procedural_intent.get("risk_tags", []),
                "expected_behavior": procedural_intent.get("expected_behavior", ""),
            }
        )
    mutation_support = Counter(item["category"] for item in retrieved)
    intents.append({"name": "mutation_coverage_summary", "risk_tags": list(mutation_support.keys()), "expected_behavior": "Retrieved mutations should cover multiple fault categories."})
    return intents

def build_test_cases(schema: dict, retrieved_mutations: list[dict], planned_intents: list[dict]) -> list[dict]:
    seeds = build_seed_rows(schema)
    cases = []
    for entity in schema.get("entities", []):
        cases.append({
            "id": f"normal_{entity['name'].lower()}",
            "category": "normal",
            "title": f"Normal data load for {entity['name']}",
            "setup_sql": [render_insert(entity["name"], row) for row in seeds[entity["name"]][:2]],
            "assertions": [f"Ensure {entity['name']} loads successfully."],
        })
    for entity in schema.get("entities", []):
        if not entity.get("attributes"):
            continue
        first_attr = entity["attributes"][0]
        boundary_row = dict(seeds[entity["name"]][0])
        if "NUMBER" in first_attr.get("type_hint", "").upper() or "INT" in first_attr.get("type_hint", "").upper():
            boundary_row[first_attr["name"]] = 0
        cases.append({
            "id": f"boundary_{entity['name'].lower()}",
            "category": "boundary",
            "title": f"Boundary check for {entity['name']}",
            "setup_sql": [render_insert(entity["name"], boundary_row)],
            "assertions": [f"Validate boundary behavior for {entity['name']}."],
        })
        negative_row = dict(seeds[entity["name"]][0])
        negative_row[first_attr["name"]] = None
        cases.append({
            "id": f"negative_{entity['name'].lower()}",
            "category": "negative",
            "title": f"Negative input for {entity['name']}",
            "setup_sql": [render_insert(entity["name"], negative_row)],
            "assertions": [f"Expect controlled rejection or validation for {entity['name']}."],
        })
    anchor = schema["entities"][0]["name"] if schema.get("entities") else "TARGET"
    anchor_row = dict(seeds.get(anchor, [{}])[0]) if seeds.get(anchor) else {}
    for index, mutation in enumerate(retrieved_mutations[:4], start=1):
        mutated_row = dict(anchor_row)
        mutation_terms = {canonical_focus_keyword(token) for token in tokenize(mutation["mutation_name"] + " " + " ".join(mutation.get("tags", [])))}
        for key in list(mutated_row.keys()):
            lower_key = key.lower()
            if "withdrawal" in mutation_terms and "bal" in lower_key:
                mutated_row[key] = 100.0
            if "threshold" in mutation_terms and ("type" in lower_key or "bal" in lower_key):
                mutated_row[key] = 0 if "bal" in lower_key else "B"
        cases.append({
            "id": f"mutation_{index:02d}",
            "category": "mutation",
            "title": mutation["mutation_name"],
            "setup_sql": [render_insert(anchor, mutated_row)] if mutated_row else [],
            "assertions": [mutation.get("expected_observable_effect", "Mutation should be revealed by this case.")],
            "mutation_id": mutation["id"],
        })
    if any(intent["name"] == "safe_withdrawal" for intent in planned_intents):
        cases.append({
            "id": "exception_insufficient_funds",
            "category": "exception",
            "title": "Insufficient funds rejection",
            "setup_sql": [],
            "assertions": ["Withdrawal should fail safely when the requested amount exceeds the balance."],
        })
    return cases

planned_test_intents = plan_test_intents(schema, retrieved_mutations)
test_cases = build_test_cases(schema, retrieved_mutations, planned_test_intents)
print("planned_test_intents:", json.dumps(planned_test_intents, indent=2))
print("generated_test_case_count:", len(test_cases))


planned_test_intents: [
  {
    "name": "threshold_classification",
    "risk_tags": [
      "boundary",
      "comparison",
      "customer",
      "threshold"
    ],
    "expected_behavior": "Customers above the threshold should move to class A and other customers should move to class B."
  },
  {
    "name": "safe_withdrawal",
    "risk_tags": [
      "boundary",
      "transaction",
      "withdrawal",
      "balance"
    ],
    "expected_behavior": "Exact-balance withdrawals should work and insufficient funds should be rejected safely."
  },
  {
    "name": "branch_transfer",
    "risk_tags": [
      "branch",
      "transfer",
      "transaction",
      "foreign_key"
    ],
    "expected_behavior": "Dependent accounts should move before the source branch is deleted."
  },
  {
    "name": "mutation_coverage_summary",
    "risk_tags": [
      "transaction",
      "exception",
      "business_rule",
      "parameter",
      "relational",
      "control_flow"
    ],
    "expected_beh


## Stage 6: Submission Analysis And Rubric Scoring

This stage uses static PL/SQL signals to produce a more nuanced rubric. The scoring now considers case categories, procedural coverage, and exception-related readiness.


In [7]:

def analyze_submission(submission: str) -> dict:
    upper = submission.upper()
    return {
        "has_if": "IF" in upper,
        "has_update": "UPDATE" in upper,
        "has_delete": "DELETE" in upper,
        "has_exception": "EXCEPTION" in upper,
        "has_transaction_control": ("COMMIT" in upper or "ROLLBACK" in upper),
        "has_raise_application_error": "RAISE_APPLICATION_ERROR" in upper,
    }

def score_submission(test_cases: list[dict], analysis: dict) -> dict:
    breakdown = {
        "normal_cases": 20.0,
        "boundary_cases": 20.0,
        "negative_cases": 15.0,
        "mutation_cases": 20.0,
        "procedural_logic": 15.0,
        "exception_handling": 10.0,
    }
    if not analysis["has_if"]:
        breakdown["procedural_logic"] -= 6.0
    if not analysis["has_update"]:
        breakdown["procedural_logic"] -= 5.0
    if not analysis["has_transaction_control"]:
        breakdown["procedural_logic"] -= 4.0
    if not analysis["has_exception"]:
        breakdown["exception_handling"] -= 5.0
    if not analysis["has_raise_application_error"]:
        breakdown["exception_handling"] -= 2.0
    total = sum(max(0.0, value) for value in breakdown.values())
    feedback = []
    if not analysis["has_transaction_control"]:
        feedback.append("Submission is missing explicit transaction control for risky account updates.")
    if not analysis["has_exception"]:
        feedback.append("Submission should expose safer exception handling paths.")
    if not analysis["has_raise_application_error"]:
        feedback.append("Submission could use more explicit rejection signaling for invalid operations.")
    if not feedback:
        feedback.append("Submission includes the core procedural constructs expected by the rubric.")
    return {
        "total_score": round(total, 2),
        "breakdown": {key: round(max(0.0, value), 2) for key, value in breakdown.items()},
        "feedback": feedback,
    }

submission_analysis = analyze_submission(submission_text)
rubric_score = score_submission(test_cases, submission_analysis)
print("Submission analysis:", submission_analysis)
print("Rubric score:", json.dumps(rubric_score, indent=2))


Submission analysis: {'has_if': True, 'has_update': True, 'has_delete': False, 'has_exception': True, 'has_transaction_control': True, 'has_raise_application_error': True}
Rubric score: {
  "total_score": 100.0,
  "breakdown": {
    "normal_cases": 20.0,
    "boundary_cases": 20.0,
    "negative_cases": 15.0,
    "mutation_cases": 20.0,
    "procedural_logic": 15.0,
    "exception_handling": 10.0
  },
  "feedback": [
    "Submission includes the core procedural constructs expected by the rubric."
  ]
}



## Generation Run

This final execution stage saves the schema SQL, synthesized test data, RAG evaluation outputs, and rubric results so the notebook can act as a complete project artifact rather than only an interactive scratchpad.


In [8]:

test_data_sql = []
for case in test_cases:
    test_data_sql.extend(case["setup_sql"])

(DATA_DIR / "Schema.sql").write_text(ddl_text, encoding="utf-8")
(DATA_DIR / "Test_data.sql").write_text("\n".join(test_data_sql), encoding="utf-8")
(ARTIFACT_DIR / "corpus_profile.json").write_text(
    json.dumps(
        {
            "document_count": corpus_profile["document_count"],
            "category_distribution": dict(corpus_profile["category_distribution"]),
            "avg_tag_count": corpus_profile["avg_tag_count"],
        },
        indent=2,
    ),
    encoding="utf-8",
)
(ARTIFACT_DIR / "retrieved_mutations.json").write_text(json.dumps(retrieved_mutations, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "rag_quality.json").write_text(json.dumps(rag_quality, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "planned_test_intents.json").write_text(json.dumps(planned_test_intents, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "test_cases.json").write_text(json.dumps(test_cases, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "rubric_score.json").write_text(json.dumps(rubric_score, indent=2), encoding="utf-8")

print("Top retrieved mutations:")
for item in retrieved_mutations:
    print("-", item["id"], item["mutation_name"], "|", item["category"], "|", round(item["total_score"], 3))
print("\nRAG quality:", rag_quality)
print("Generated test cases:", len(test_cases))
print("Submission analysis:", submission_analysis)
print("Rubric score:", rubric_score["total_score"])
for line in rubric_score["feedback"]:
    print("-", line)


Top retrieved mutations:
- MUT-043 COMMIT Removal | transaction | 5.03
- MUT-051 SQLCODE Branch Mutation | exception | 7.47
- MUT-074 Threshold Tightening | business_rule | 5.399
- MUT-050 Raise Removed | exception | 6.663
- MUT-009 Call-Site Boundary Value | parameter | 6.962
- MUT-014 Greater-Than to Greater-Or-Equal | relational | 6.949
- MUT-020 IF Branch Removal | control_flow | 5.667
- MUT-028 GOTO Target Mutation | control_flow | 5.548

RAG quality: {'intent_proxy_coverage': 1.0, 'intent_hits': {'threshold_classification': True, 'safe_withdrawal': True, 'branch_transfer': True}, 'retrieved_categories': {'transaction': 1, 'exception': 2, 'business_rule': 1, 'parameter': 1, 'relational': 1, 'control_flow': 2}, 'retrieved_count': 8, 'diverse_category_count': 6}
Generated test cases: 14
Submission analysis: {'has_if': True, 'has_update': True, 'has_delete': False, 'has_exception': True, 'has_transaction_control': True, 'has_raise_application_error': True}
Rubric score: 100.0
- Submi


## Testing Checkpoint

These tests validate the upgraded pipeline end to end. They specifically test the RAG quality stage, the diversity of generated test categories, and the presence of saved outputs that support the rest of the project.


In [9]:

mutation_names = " ".join(item["mutation_name"] for item in retrieved_mutations)
categories = {case["category"] for case in test_cases}
assert corpus_profile["document_count"] >= 80, "The existing mutation dataset should contain at least 80 documents."
assert rag_quality["intent_proxy_coverage"] >= 0.66, "RAG intent-proxy coverage is too low for this assignment."
assert all(rag_quality["intent_hits"].values()), "At least one procedural intent is not supported by the retrieved mutation set."
assert rag_quality["diverse_category_count"] >= 3, "Retrieved mutation set should cover multiple categories."
assert ("Withdrawal" in mutation_names or "Branch" in mutation_names or "Threshold" in mutation_names), "Expected bank-relevant mutations were not retrieved."
assert {"normal", "boundary", "negative", "mutation", "exception"}.issubset(categories), "Missing one or more required test-case categories."
assert (DATA_DIR / "Schema.sql").exists(), "Schema.sql was not written."
assert (DATA_DIR / "Test_data.sql").exists(), "Test_data.sql was not written."
assert rubric_score["total_score"] > 0, "Rubric score should be positive."
print("Test-case notebook tests passed.")


Test-case notebook tests passed.
